In [32]:
import os
import random
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

from lightgbm import LGBMClassifier
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

Reprodutibilidade

In [2]:
GERME_ALEATORIO = 42

os.environ["PYTHONHASHSEED"] = str(GERME_ALEATORIO)

random.seed(GERME_ALEATORIO)
np.random.seed(GERME_ALEATORIO)

print("Germe aleatório configurado:", GERME_ALEATORIO)

Germe aleatório configurado: 42


Localiza e carrega dataset

In [4]:
CAMINHO_DATASET = Path("/content/dataset2_sem_duplicatas.csv")

if not CAMINHO_DATASET.exists():
    CAMINHO_DATASET = Path("/content/dataset2_sem_duplicatas.csv")

if not CAMINHO_DATASET.exists():
    raise FileNotFoundError(
        "Dataset não encontrado. Verifique a pasta Dataset."
    )

df = pd.read_csv(
    CAMINHO_DATASET,
    encoding="utf-8-sig"
)

print("Dataset carregado:", CAMINHO_DATASET)
print("Dimensões:", df.shape)

display(df.head())

Dataset carregado: /content/dataset2_sem_duplicatas.csv
Dimensões: (3397, 4)


,id,fonte,texto,classe
0,1_0,sms,"Oi, peço para transferir aquele valor para meu...",Legitimate
1,2_0,sms,Bom dia babe..tudo bem? Não se esqueça de liga...,Legitimate
2,3_0,sms,AEN8AFWXJHC Confirmado. Compraste 19.00MT de c...,Legitimate
3,4_0,sms,"7GD04E51YZM. Caro Cliente, o codigo para efect...",Legitimate
4,5_0,sms,Bom dia babe. Está bem. Vou comprar. Boa viagem,Legitimate


Validação antes da modelagem:

In [5]:
COLUNA_TEXTO = "texto"
COLUNA_ALVO = "classe"

colunas_necessarias = {
    "id",
    "fonte",
    COLUNA_TEXTO,
    COLUNA_ALVO
}

colunas_ausentes = colunas_necessarias - set(df.columns)

if colunas_ausentes:
    raise ValueError(
        f"Colunas ausentes no dataset: {colunas_ausentes}"
    )

print("Valores ausentes:")
print(df.isnull().sum())

print("\nDuplicatas de texto:")
print(df.duplicated(subset=[COLUNA_TEXTO]).sum())

print("\nDistribuição das classes:")
print(df[COLUNA_ALVO].value_counts())

print("\nProporção das classes:")
print(df[COLUNA_ALVO].value_counts(normalize=True))

Valores ausentes:
id        0
fonte     0
texto     0
classe    0
dtype: int64

Duplicatas de texto:
0

Distribuição das classes:
classe
Legitimate    2006
Smishing      1391
Name: count, dtype: int64

Proporção das classes:
classe
Legitimate    0.590521
Smishing      0.409479
Name: proportion, dtype: float64


In [6]:
if df[COLUNA_TEXTO].isnull().any():
    raise ValueError("Existem textos ausentes.")

if df[COLUNA_ALVO].isnull().any():
    raise ValueError("Existem classes ausentes.")

if df.duplicated(subset=[COLUNA_TEXTO]).any():
    raise ValueError("Ainda existem textos duplicados.")

classes_esperadas = {"Legitimate", "Smishing"}

if set(df[COLUNA_ALVO].unique()) != classes_esperadas:
    raise ValueError(
        f"Classes inesperadas: {df[COLUNA_ALVO].unique()}"
    )

print("Dataset validado e pronto para modelagem.")

Dataset validado e pronto para modelagem.


In [7]:
MAPA_CLASSES = {
    "Legitimate": 0,
    "Smishing": 1
}

df_modelagem = df.copy()

df_modelagem["alvo"] = (
    df_modelagem[COLUNA_ALVO]
    .map(MAPA_CLASSES)
)

if df_modelagem["alvo"].isnull().any():
    raise ValueError(
        "Alguma classe não foi convertida corretamente."
    )

print(df_modelagem[[COLUNA_ALVO, "alvo"]].drop_duplicates())

          classe  alvo
0     Legitimate     0
2006    Smishing     1


Primeira divisão: 70% treino e 30% temporário:

In [8]:
df_treino, df_temporario = train_test_split(
    df_modelagem,
    test_size=0.30,
    random_state=GERME_ALEATORIO,
    stratify=df_modelagem["alvo"]
)

print("Treino:", len(df_treino))
print("Temporário:", len(df_temporario))

Treino: 2377
Temporário: 1020


Segunda divisão: 15% validação e 15% teste

In [9]:
df_validacao, df_teste = train_test_split(
    df_temporario,
    test_size=0.50,
    random_state=GERME_ALEATORIO,
    stratify=df_temporario["alvo"]
)

print("Treino:", len(df_treino))
print("Validação:", len(df_validacao))
print("Teste:", len(df_teste))

print("\nTotal:", (
    len(df_treino) +
    len(df_validacao) +
    len(df_teste)
))

Treino: 2377
Validação: 510
Teste: 510

Total: 3397


Conferir proporções

In [10]:
def mostrar_distribuicao(nome, dados):
    print(f"\n{nome}")
    print("-" * len(nome))
    print("Quantidade:", len(dados))

    print(
        dados["classe"]
        .value_counts()
    )

    print(
        dados["classe"]
        .value_counts(normalize=True)
        .round(4)
    )


mostrar_distribuicao("Treino", df_treino)
mostrar_distribuicao("Validação", df_validacao)
mostrar_distribuicao("Teste", df_teste)


Treino
------
Quantidade: 2377
classe
Legitimate    1404
Smishing       973
Name: count, dtype: int64
classe
Legitimate    0.5907
Smishing      0.4093
Name: proportion, dtype: float64

Validação
---------
Quantidade: 510
classe
Legitimate    301
Smishing      209
Name: count, dtype: int64
classe
Legitimate    0.5902
Smishing      0.4098
Name: proportion, dtype: float64

Teste
-----
Quantidade: 510
classe
Legitimate    301
Smishing      209
Name: count, dtype: int64
classe
Legitimate    0.5902
Smishing      0.4098
Name: proportion, dtype: float64


Verificar se não existe sobreposição

In [11]:
ids_treino = set(df_treino["id"])
ids_validacao = set(df_validacao["id"])
ids_teste = set(df_teste["id"])

assert ids_treino.isdisjoint(ids_validacao)
assert ids_treino.isdisjoint(ids_teste)
assert ids_validacao.isdisjoint(ids_teste)

print("Não existe sobreposição de IDs entre os conjuntos.")

Não existe sobreposição de IDs entre os conjuntos.


Salvar as divisões

In [12]:
PASTA_DATASET = CAMINHO_DATASET.parent

df_treino.to_csv(
    PASTA_DATASET / "treino.csv",
    index=False,
    encoding="utf-8-sig"
)

df_validacao.to_csv(
    PASTA_DATASET / "validacao.csv",
    index=False,
    encoding="utf-8-sig"
)

df_teste.to_csv(
    PASTA_DATASET / "teste.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Divisões salvas em:", PASTA_DATASET)

Divisões salvas em: /content


Define X e Y

In [13]:
X_treino = df_treino["texto"].astype(str)
y_treino = df_treino["alvo"].astype(int)

X_validacao = df_validacao["texto"].astype(str)
y_validacao = df_validacao["alvo"].astype(int)

X_teste = df_teste["texto"].astype(str)
y_teste = df_teste["alvo"].astype(int)

print("X treino:", X_treino.shape)
print("y treino:", y_treino.shape)

print("X validação:", X_validacao.shape)
print("y validação:", y_validacao.shape)

print("X teste:", X_teste.shape)
print("y teste:", y_teste.shape)

X treino: (2377,)
y treino: (2377,)
X validação: (510,)
y validação: (510,)
X teste: (510,)
y teste: (510,)


Criar a Árvore baseline

In [14]:
pipeline_baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "arvore",
        DecisionTreeClassifier(
            random_state=GERME_ALEATORIO
        )
    )
])

print(pipeline_baseline)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('arvore', DecisionTreeClassifier(random_state=42))])


Treinar a baseline

In [15]:
pipeline_baseline.fit(
    X_treino,
    y_treino
)

print("Árvore baseline treinada.")

Árvore baseline treinada.


Diagnóstico inicial de treinamento

In [16]:
pred_treino_baseline = pipeline_baseline.predict(X_treino)
pred_validacao_baseline = pipeline_baseline.predict(X_validacao)

f1_treino_baseline = f1_score(
    y_treino,
    pred_treino_baseline
)

f1_validacao_baseline = f1_score(
    y_validacao,
    pred_validacao_baseline
)

print(
    f"F1 no treino: {f1_treino_baseline:.4f}"
)

print(
    f"F1 na validação: {f1_validacao_baseline:.4f}"
)

print(
    "Diferença:",
    round(
        f1_treino_baseline -
        f1_validacao_baseline,
        4
    )
)

F1 no treino: 0.9995
F1 na validação: 0.9294
Diferença: 0.07


In [17]:
#Complexidade da árvore:
arvore_baseline = (
    pipeline_baseline
    .named_steps["arvore"]
)

print(
    "Profundidade:",
    arvore_baseline.get_depth()
)

print(
    "Quantidade de folhas:",
    arvore_baseline.get_n_leaves()
)

Profundidade: 43
Quantidade de folhas: 89


Ajustar hiperparâmetros

In [18]:
GRADE_PARAMETROS = {
    "arvore__criterion": [
        "gini",
        "entropy"
    ],

    "arvore__max_depth": [
        10,
        20,
        30,
        None
    ],

    "arvore__min_samples_split": [
        2,
        5,
        10
    ],

    "arvore__min_samples_leaf": [
        1,
        2,
        5
    ],

    "arvore__class_weight": [
        None,
        "balanced"
    ]
}

In [27]:
busca_grade = GridSearchCV(
    estimator=pipeline_baseline,
    param_grid=GRADE_PARAMETROS,
    scoring="f1",
    cv=2,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

In [28]:
busca_grade.fit(
    X_treino,
    y_treino
)

print("Busca concluída.")

print("\nMelhores hiperparâmetros:")
print(busca_grade.best_params_)

print("\nMelhor F1 médio na validação cruzada:")
print(round(busca_grade.best_score_, 4))

Fitting 2 folds for each of 144 candidates, totalling 288 fits
Busca concluída.

Melhores hiperparâmetros:
{'arvore__class_weight': None, 'arvore__criterion': 'entropy', 'arvore__max_depth': 10, 'arvore__min_samples_leaf': 1, 'arvore__min_samples_split': 2}

Melhor F1 médio na validação cruzada:
0.9123


-> O resultado acima mostra que a árvore simples praticamente decorou o treinamento.
O F1 de validação ainda é bom, mas existe uma diferença de sete pontos percentuais, sugerindo algum overfitting.

In [29]:
melhor_arvore = busca_grade.best_estimator_

pred_validacao_arvore_ajustada = melhor_arvore.predict(
    X_validacao
)

f1_validacao_arvore_ajustada = f1_score(
    y_validacao,
    pred_validacao_arvore_ajustada
)

print(
    "F1 da Árvore de Decisão ajustada na validação:",
    round(f1_validacao_arvore_ajustada, 4)
)

F1 da Árvore de Decisão ajustada na validação: 0.9133


Instalar o LightGBM

In [30]:
%pip install lightgbm

LightGBM baseline - Criando pipeline inicial. Assim como na Árvore de Decisão Simples, coloque o TF-IDF dentro do pipeline para evitar vazamento

In [33]:
pipeline_lightgbm_baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "lightgbm",
        LGBMClassifier(
            objective="binary",
            random_state=GERME_ALEATORIO,
            n_estimators=100,
            learning_rate=0.1,
            n_jobs=-1,
            verbosity=-1
        )
    )
])

print(pipeline_lightgbm_baseline)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('lightgbm',
                 LGBMClassifier(n_jobs=-1, objective='binary', random_state=42,
                                verbosity=-1))])


Treinar o LightGBM baseline

In [34]:
pipeline_lightgbm_baseline.fit(
    X_treino,
    y_treino
)

print("LightGBM baseline treinado.")

LightGBM baseline treinado.


Diagnóstico inicial do LightGBM com base no treino:

In [35]:
pred_treino_lightgbm = (
    pipeline_lightgbm_baseline.predict(X_treino)
)

pred_validacao_lightgbm = (
    pipeline_lightgbm_baseline.predict(X_validacao)
)

f1_treino_lightgbm = f1_score(
    y_treino,
    pred_treino_lightgbm
)

f1_validacao_lightgbm = f1_score(
    y_validacao,
    pred_validacao_lightgbm
)

print(
    f"F1 no treino: {f1_treino_lightgbm:.4f}"
)

print(
    f"F1 na validação: {f1_validacao_lightgbm:.4f}"
)

print(
    "Diferença entre treino e validação:",
    round(
        f1_treino_lightgbm -
        f1_validacao_lightgbm,
        4
    )
)

F1 no treino: 0.9969
F1 na validação: 0.9538
Diferença entre treino e validação: 0.0431


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Comparação com árvore simples:

In [36]:
comparacao_inicial = pd.DataFrame({
    "modelo": [
        "Árvore simples",
        "Árvore ajustada",
        "LightGBM baseline"
    ],
    "f1_validacao": [
        f1_validacao_baseline,
        f1_validacao_arvore_ajustada,
        f1_validacao_lightgbm
    ]
})

display(comparacao_inicial)

,modelo,f1_validacao
0,Árvore simples,0.929440
1,Árvore ajustada,0.913265
2,LightGBM baseline,0.953771


### Comparação inicial dos modelos baseados em árvores

A Árvore de Decisão simples obteve F1 de 0,9294 na validação, enquanto sua versão ajustada obteve 0,9133. Portanto, a otimização realizada não melhorou a generalização da árvore individual.

O LightGBM baseline obteve F1 de 0,9538 na validação, superando as duas versões da Árvore de Decisão. A diferença entre o F1 de treino e validação também caiu de 0,0700, na árvore simples, para 0,0431 no LightGBM. Isso indica uma redução do sobreajuste, embora ainda exista diferença entre o desempenho nos dados conhecidos e nos dados de validação.

Diante desses resultados, o LightGBM será refinado por meio do ajuste de hiperparâmetros. O conjunto de teste permanecerá isolado até a escolha definitiva do modelo.

Ajustar os hiperparâmetros do LightGBM - Criando novo pipeline para isso

In [37]:
pipeline_lightgbm_otimizacao = Pipeline([
    (
        "tfidf",
        TfidfVectorizer()
    ),
    (
        "lightgbm",
        LGBMClassifier(
            objective="binary",
            random_state=GERME_ALEATORIO,
            n_jobs=1,
            verbosity=-1,
            subsample_freq=1
        )
    )
])

Definir os hiperparâmetros

In [38]:
PARAMETROS_LIGHTGBM = {
    "tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],

    "tfidf__min_df": [
        1,
        2,
        5
    ],

    "tfidf__max_features": [
        5000,
        10000,
        None
    ],

    "lightgbm__n_estimators": [
        100,
        200,
        300,
        500
    ],

    "lightgbm__learning_rate": [
        0.02,
        0.05,
        0.1
    ],

    "lightgbm__num_leaves": [
        7,
        15,
        31,
        63
    ],

    "lightgbm__max_depth": [
        5,
        10,
        15,
        -1
    ],

    "lightgbm__min_child_samples": [
        10,
        20,
        40,
        60
    ],

    "lightgbm__subsample": [
        0.7,
        0.8,
        1.0
    ],

    "lightgbm__colsample_bytree": [
        0.6,
        0.8,
        1.0
    ],

    "lightgbm__reg_alpha": [
        0.0,
        0.1,
        1.0
    ],

    "lightgbm__reg_lambda": [
        0.0,
        0.1,
        1.0
    ],

    "lightgbm__class_weight": [
        None,
        "balanced"
    ]
}

Configurar validação cruzada

In [39]:
validacao_cruzada = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=GERME_ALEATORIO
)

Configurar métricas da busca

In [40]:
metricas_busca = {
    "f1": "f1",
    "recall": "recall",
    "precisao": "precision"
}

In [41]:
busca_lightgbm = RandomizedSearchCV(
    estimator=pipeline_lightgbm_otimizacao,
    param_distributions=PARAMETROS_LIGHTGBM,
    n_iter=40,
    scoring=metricas_busca,
    refit="f1",
    cv=validacao_cruzada,
    random_state=GERME_ALEATORIO,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

In [42]:
busca_lightgbm.fit(
    X_treino,
    y_treino
)

print("Busca concluída.")

print("\nMelhores hiperparâmetros:")
print(busca_lightgbm.best_params_)

print("\nMelhor F1 médio na validação cruzada:")
print(round(busca_lightgbm.best_score_, 4))

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Busca concluída.

Melhores hiperparâmetros:
{'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'tfidf__max_features': 5000, 'lightgbm__subsample': 0.8, 'lightgbm__reg_lambda': 0.1, 'lightgbm__reg_alpha': 1.0, 'lightgbm__num_leaves': 63, 'lightgbm__n_estimators': 200, 'lightgbm__min_child_samples': 10, 'lightgbm__max_depth': -1, 'lightgbm__learning_rate': 0.1, 'lightgbm__colsample_bytree': 0.6, 'lightgbm__class_weight': 'balanced'}

Melhor F1 médio na validação cruzada:
0.9517


Recuperar os resultados da melhor configuração

In [43]:
indice_melhor = busca_lightgbm.best_index_

melhor_f1_cv = (
    busca_lightgbm.cv_results_["mean_test_f1"][indice_melhor]
)

melhor_recall_cv = (
    busca_lightgbm.cv_results_["mean_test_recall"][indice_melhor]
)

melhor_precisao_cv = (
    busca_lightgbm.cv_results_["mean_test_precisao"][indice_melhor]
)

print("Resultados médios da melhor configuração:")

print(
    "F1:",
    round(melhor_f1_cv, 4)
)

print(
    "Recall:",
    round(melhor_recall_cv, 4)
)

print(
    "Precisão:",
    round(melhor_precisao_cv, 4)
)

Resultados médios da melhor configuração:
F1: 0.9517
Recall: 0.9424
Precisão: 0.9613


Avaliar o LightGBM ajustado na validação

In [44]:
melhor_lightgbm = busca_lightgbm.best_estimator_

pred_treino_lightgbm_ajustado = (
    melhor_lightgbm.predict(X_treino)
)

pred_validacao_lightgbm_ajustado = (
    melhor_lightgbm.predict(X_validacao)
)

f1_treino_lightgbm_ajustado = f1_score(
    y_treino,
    pred_treino_lightgbm_ajustado
)

f1_validacao_lightgbm_ajustado = f1_score(
    y_validacao,
    pred_validacao_lightgbm_ajustado
)

diferenca_lightgbm_ajustado = (
    f1_treino_lightgbm_ajustado -
    f1_validacao_lightgbm_ajustado
)

print(
    "F1 treino:",
    round(f1_treino_lightgbm_ajustado, 4)
)

print(
    "F1 validação:",
    round(f1_validacao_lightgbm_ajustado, 4)
)

print(
    "Diferença:",
    round(diferenca_lightgbm_ajustado, 4)
)

F1 treino: 0.9954
F1 validação: 0.9662
Diferença: 0.0292


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Comparar todas as versões

In [45]:
f1_treino_arvore_ajustada = f1_score(
    y_treino,
    melhor_arvore.predict(X_treino)
)

comparacao_final_modelagem = pd.DataFrame({
    "modelo": [
        "Árvore simples",
        "Árvore ajustada",
        "LightGBM baseline",
        "LightGBM ajustado"
    ],

    "f1_treino": [
        f1_treino_baseline,
        f1_treino_arvore_ajustada,
        f1_treino_lightgbm,
        f1_treino_lightgbm_ajustado
    ],

    "f1_validacao": [
        f1_validacao_baseline,
        f1_validacao_arvore_ajustada,
        f1_validacao_lightgbm,
        f1_validacao_lightgbm_ajustado
    ]
})

comparacao_final_modelagem["diferenca"] = (
    comparacao_final_modelagem["f1_treino"] -
    comparacao_final_modelagem["f1_validacao"]
)

display(
    comparacao_final_modelagem.round(4)
)

,modelo,f1_treino,f1_validacao,diferenca
0,Árvore simples,0.9995,0.9294,0.0700
1,Árvore ajustada,0.9552,0.9133,0.0420
2,LightGBM baseline,0.9969,0.9538,0.0431
3,LightGBM ajustado,0.9954,0.9662,0.0292


Como escolher entre baseline e ajustado

In [46]:
if (
    f1_validacao_lightgbm_ajustado
    >
    f1_validacao_lightgbm
):
    modelo_selecionado = melhor_lightgbm
    nome_modelo_selecionado = "LightGBM ajustado"
else:
    modelo_selecionado = pipeline_lightgbm_baseline
    nome_modelo_selecionado = "LightGBM baseline"

print(
    "Modelo selecionado:",
    nome_modelo_selecionado
)

Modelo selecionado: LightGBM ajustado


Salvar os modelos

In [47]:
import joblib
from pathlib import Path

PASTA_RESULTADOS = Path("resultados")

PASTA_RESULTADOS.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    pipeline_baseline,
    PASTA_RESULTADOS / "arvore_simples.joblib"
)

joblib.dump(
    melhor_arvore,
    PASTA_RESULTADOS / "arvore_ajustada.joblib"
)

joblib.dump(
    pipeline_lightgbm_baseline,
    PASTA_RESULTADOS / "lightgbm_baseline.joblib"
)

joblib.dump(
    melhor_lightgbm,
    PASTA_RESULTADOS / "lightgbm_ajustado.joblib"
)

joblib.dump(
    modelo_selecionado,
    PASTA_RESULTADOS / "modelo_selecionado.joblib"
)

print("Modelos salvos em:")
print(PASTA_RESULTADOS.resolve())

Modelos salvos em:
/content/resultados


Salvar os hiperparâmetros e resultados

In [49]:
import json

informacoes_modelagem = {
    "germe_aleatorio": GERME_ALEATORIO,

    "divisao": {
        "treino": len(X_treino),
        "validacao": len(X_validacao),
        "teste": len(X_teste)
    },

    "f1_validacao": {
        "arvore_simples": float(
            f1_validacao_baseline
        ),
        "arvore_ajustada": float(
            f1_validacao_arvore_ajustada
        ),
        "lightgbm_baseline": float(
            f1_validacao_lightgbm
        ),
        "lightgbm_ajustado": float(
            f1_validacao_lightgbm_ajustado
        )
    },

    "melhores_hiperparametros_lightgbm": (
        busca_lightgbm.best_params_
    ),

    "melhor_f1_validacao_cruzada": float(
        busca_lightgbm.best_score_
    ),

    "modelo_selecionado": (
        nome_modelo_selecionado
    )
}

In [50]:
with open(
    PASTA_RESULTADOS / "informacoes_modelagem.json",
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        informacoes_modelagem,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

print("Informações da modelagem salvas.")

Informações da modelagem salvas.
